# 02.4 正则化与学习率调度（Regularization and Scheduler）

这一节关注两个训练中的关键问题：  

1. 怎么减少过拟合
2. 学习率怎么随训练过程变化

重点概念

- 过拟合（overfitting）
- 正则化（regularization）
- 权重衰减（weight decay）
- 学习率调度器（learning rate scheduler）


## 学习目标

学完后你应该能

1. 用训练曲线识别过拟合
2. 理解 Dropout 的训练
3. weight decay 的作用
4. 使用 `StepLR` 调整学习率
5. 对比无正则化和有正则化模型
6. 把这些技巧迁移到后续更大的项目中

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

## 1. 准备数据

为了把重点放在训练行为上，这里用 `digits` 数据集并把图像展平，交给 MLP。  


In [ ]:
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32).reshape(-1, 64) / 16.0
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print("X_train.shape =", X_train.shape)
print("X_val.shape =", X_val.shape)

## 2. Dropout 的直觉

Dropout 的核心思想是：训练时随机“关掉”一部分神经元输出。  

这样做的目的

- 降低神经元之间的过度依赖（reduce over-dependence between neurons）
- 提高泛化能力（improve generalization）

In [ ]:
torch.manual_seed(0)

drop = nn.Dropout(p=0.5)
x = torch.ones(1, 8)

drop.train()
out_train_1 = drop(x)
out_train_2 = drop(x)

drop.eval()
out_eval_1 = drop(x)
out_eval_2 = drop(x)

print("train output 1 =", out_train_1)
print("train output 2 =", out_train_2)
print("eval output 1 =", out_eval_1)
print("eval output 2 =", out_eval_2)

注意观察

- `train()` 模式下，每次输出可能不同（outputs may differ each time in `train()` mode）
- `eval()` 模式下，输出稳定（outputs are stable in `eval()` mode）

这就是为什么评估和推理时必须切到 `model.eval()`。  


## 3. 一个可选 Dropout 的 MLP

我们用一个小型 MLP 来比较：  

- 无 Dropout（no Dropout）
- 有 Dropout（with Dropout）

In [ ]:
class DigitsMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


baseline_model = DigitsMLP(dropout_p=0.0)
regularized_model = DigitsMLP(dropout_p=0.3)

print(baseline_model)
print()
print(regularized_model)

## 4. 训练与评估函数

这里复用一个简单训练框架，同时记录学习率。  


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def train_model(model, train_loader, val_loader, loss_fn, optimizer, scheduler=None, epochs=6):
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        current_lr = optimizer.param_groups[0]["lr"]

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr": current_lr,
            }
        )

        if scheduler is not None:
            scheduler.step()

    return pd.DataFrame(history)

## 5. 先训练一个无正则化基线

这里的设置

- 无 Dropout（no Dropout）
- 无权重衰减（no weight decay）
- 无学习率调度器（no scheduler）

In [ ]:
torch.manual_seed(0)
baseline = DigitsMLP(dropout_p=0.0)
loss_fn = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline.parameters(), lr=0.01)

baseline_history = train_model(
    baseline,
    train_loader,
    val_loader,
    loss_fn,
    baseline_optimizer,
    scheduler=None,
    epochs=6,
)

print(baseline_history)

## 6. 加入 Dropout、Weight Decay 和 Scheduler

这次的设置

- `dropout_p=0.3`
- `weight_decay=1e-3`
- `StepLR(step_size=3, gamma=0.5)`

In [ ]:
torch.manual_seed(0)
regularized = DigitsMLP(dropout_p=0.3)
regularized_optimizer = torch.optim.Adam(regularized.parameters(), lr=0.01, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(regularized_optimizer, step_size=3, gamma=0.5)

regularized_history = train_model(
    regularized,
    train_loader,
    val_loader,
    loss_fn,
    regularized_optimizer,
    scheduler=scheduler,
    epochs=6,
)

print(regularized_history)

## 7. 对比结果

这里不追求“一定谁更强”，而是观察这些技巧如何影响训练行为。  


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["baseline", "regularized"],
        "final_train_loss": [baseline_history.iloc[-1]["train_loss"], regularized_history.iloc[-1]["train_loss"]],
        "final_train_acc": [baseline_history.iloc[-1]["train_acc"], regularized_history.iloc[-1]["train_acc"]],
        "final_val_loss": [baseline_history.iloc[-1]["val_loss"], regularized_history.iloc[-1]["val_loss"]],
        "final_val_acc": [baseline_history.iloc[-1]["val_acc"], regularized_history.iloc[-1]["val_acc"]],
    }
)

print(comparison)

In [ ]:
print("baseline learning rates / 基线学习率:")
print(baseline_history[["epoch", "lr"]])
print()
print("regularized learning rates / 正则化模型学习率:")
print(regularized_history[["epoch", "lr"]])

你应该注意到：  

- baseline 的学习率保持不变
- regularized 模型在第 4 个 epoch 后学习率下降

In [ ]:
# 练习 1
# 用一句话解释为什么 Dropout 在 train() 和 eval() 下行为不同。
# In one sentence, explain why Dropout behaves differently under train() and eval().

参考回答

因为 Dropout 训练时要随机丢弃部分激活来正则化，而评估时需要关闭这种随机性以得到稳定输出。  


In [ ]:
# 练习 2
# 把 StepLR 的 gamma 从 0.5 改成 0.1，再观察学习率变化表。
# Change StepLR gamma from 0.5 to 0.1, then observe the learning-rate table.

In [ ]:
# 练习 3
# 用一句话解释 weight decay 的直觉作用。
# In one sentence, explain the intuition behind weight decay.

参考回答

weight decay 会对过大的权重施加额外约束，从而帮助模型不要过度复杂化。  


## 8. 小结

这一节的核心不是背技巧名字，而是理解这些技巧解决什么问题。  

你现在应该能回答

1. 什么情况下会怀疑模型过拟合？
2. Dropout 为什么依赖 `train()`
3. weight decay 主要约束的是什么？
4. scheduler 为什么不直接改变 loss，而是改变学习率？

下一步建议

- 进入迁移学习 notebook，学习如何复用已有视觉模型结构